**Matrix mechanics and the finite square well.**
In standard quantum mechanics the time-independent Schrödinger equation (TISE)
is a continuous differential equation:

$$
\hat{H}\,\psi(x) = E\,\psi(x)
\qquad\text{where}\qquad
\hat{H} = -\frac{\hbar^2}{2m}\frac{d^2}{dx^2} + V(x)
$$

**Matrix mechanics** replaces this with a purely algebraic eigenvalue problem
by discretizing space into $N$ equally spaced grid points separated by $\Delta x$.
The continuous wavefunction $\psi(x)$ becomes a vector of $N$ values,
and the Hamiltonian operator $\hat{H}$ becomes an $N \times N$ matrix.
Solving the TISE then reduces to diagonalizing that matrix - no differential
equations, no analytic tricks, just linear algebra.

The system studied here is the **1-D finite square well**:

$$
V(x) = \begin{cases} 0 & 0 \leq x \leq L \\ V_0 & \text{otherwise} \end{cases}
$$

An electron confined inside the well has energy $E < V_0$ - a **bound state**.
Unlike the infinite square well, the wavefunction penetrates a short distance
into the classically forbidden region ($V > E$) before decaying exponentially.

In [ ]:
"""finite_square_well.ipynb"""

# Cell 01 - Physical parameters and spatial grid

import matplotlib.pyplot as plt
import numpy as np

# Spatial grid parameters
N_left = 15  # grid points to the left of the well (classically forbidden)
N_well = 100  # grid points spanning the well width L
N_right = 15  # grid points to the right of the well (classically forbidden)
dx = 1.0  # spatial step size (natural units)

# Physical constants (natural units: hbar = m = 1)
hbar = 1.0
m = 1.0
V0 = 0.05  # potential barrier height outside the well; E < V0 for bound states

# Build the spatial grid: x runs from -N_left to N_well + N_right (inclusive)
x = np.arange(-N_left, N_well + N_right + 1) * dx

# Define the potential: 0 inside the well, V0 in the classically forbidden regions
left_wall = 0.0
right_wall = float(N_well)
V = np.where((x >= left_wall) & (x <= right_wall), 0.0, V0)

print(f"Well width L         : {N_well * dx:.1f} (natural units)")
print(f"Total grid points    : {len(x)}")
print(f"Barrier height V0    : {V0}")
print(f"x range              : {x[0]:.1f} to {x[-1]:.1f}")

**Discretizing the kinetic energy operator.**
The second derivative in the kinetic energy term is approximated by the
**centered finite difference**:

$$
\frac{d^2\psi}{dx^2}\bigg|_i
\approx
\frac{\psi_{i-1} - 2\psi_i + \psi_{i+1}}{(\Delta x)^2}
$$

Substituting into the kinetic energy term
$-\dfrac{\hbar^2}{2m}\dfrac{d^2\psi}{dx^2}$ and defining
$\beta = \dfrac{\hbar^2}{2m(\Delta x)^2}$, each row $i$ of
the Hamiltonian matrix contributes:

$$
H_{ii}   = 2\beta + V_i
\qquad
H_{i,i\pm1} = -\beta
$$

The result is a **symmetric tridiagonal matrix** - diagonal entries hold
the kinetic + potential energy at each grid point, and off-diagonal entries
couple each point to its two neighbors.
Dirichlet boundary conditions ($\psi = 0$ at both ends) are enforced by
working only with the $N$ **interior** points and discarding the endpoints.

In [ ]:
# Cell 02 - Build the N x N Hamiltonian matrix

# Work only with interior points (endpoints are fixed at psi = 0)
x_int = x[1:-1]
V_int = V[1:-1]
N = len(x_int)

# Finite difference coefficient for kinetic energy
beta = hbar**2 / (2.0 * m * dx**2)  # = 0.5 in natural units

# Tridiagonal Hamiltonian:
#   main diagonal  : 2*beta + V(x)  (kinetic + potential at each point)
#   off-diagonals  : -beta           (coupling to nearest neighbors)
diag = 2.0 * beta + V_int
off = -beta * np.ones(N - 1)
H = np.diag(diag) + np.diag(off, 1) + np.diag(off, -1)

print(f"Interior grid points N  : {N}")
print(f"Hamiltonian matrix shape : {H.shape}  ({N} x {N})")
print(f"beta = hbar^2/(2m dx^2) : {beta}")
print(f"Diagonal (inside well)  : 2*beta + 0   = {2 * beta:.4f}")
print(f"Diagonal (outside well) : 2*beta + V0  = {2 * beta + V0:.4f}")
print(f"Off-diagonal            : -beta         = {-beta:.4f}")

**Diagonalizing the Hamiltonian.**
The matrix eigenvalue equation

$$
H\,\vec{\psi}_n = E_n\,\vec{\psi}_n
$$

is the discrete analogue of $\hat{H}\psi_n(x) = E_n\psi_n(x)$.
Each eigenvector $\vec{\psi}_n$ is the discretized wavefunction at the
$N$ interior grid points, and each eigenvalue $E_n$ is the corresponding
energy level.

`numpy.linalg.eigh` exploits the symmetry of $H$ (real, symmetric)
to return real eigenvalues in ascending order and orthonormal eigenvectors.
Only eigenstates with $E_n < V_0$ are **bound states** - the electron
cannot escape the well classically.
States with $E_n \geq V_0$ are scattering states and are not meaningful
in the context of this finite grid.

In [ ]:
# Cell 03 - Diagonalize H and identify bound states

# eigh returns eigenvalues in ascending order and orthonormal eigenvectors
energies, vecs = np.linalg.eigh(H)

# Bound states satisfy E < V0; identify them
bound_mask = energies < V0
n_bound = np.sum(bound_mask)

print(f"Total eigenstates       : {N}")
print(f"Bound states (E < V0)   : {n_bound}")
print()
print("First four bound-state energies:")
for n in range(4):
    # Analytic infinite-square-well energies for comparison
    L = N_well * dx
    E_inf = (n + 1) ** 2 * np.pi**2 * hbar**2 / (2 * m * L**2)
    print(f"  n={n + 1}: E = {energies[n]:.6f}  (infinite-well approx: {E_inf:.6f})")
print()
print("Finite well energies are lower than infinite-well values")
print("because the wavefunction can penetrate into the barrier region.")

**Probability density and normalization.**
The **probability density** of finding the electron at position $x$ is
$|\psi_n(x)|^2$.
`eigh` already returns orthonormal eigenvectors satisfying
$\sum_i |\psi_i|^2 = 1$, but this is a discrete sum rather than
the continuous integral $\int |\psi(x)|^2\,dx = 1$.
To recover the correctly normalized continuous probability density
we divide by $\sqrt{\Delta x}$, which ensures:

$$
\int_{-\infty}^{\infty} |\psi_n(x)|^2\,dx
\approx \sum_i |\psi_n(x_i)|^2\,\Delta x = 1
$$

The interior eigenvector is padded with zeros at the two boundary points
to reconstruct the full wavefunction on the original grid.

In [ ]:
# Cell 04 - Normalize wavefunction and plot probability densities

plt.figure("mm_finite_square_well", figsize=(10, 6))

for n in range(4):
    # Extract the eigenvector for energy level n (interior points only)
    psi_int = vecs[:, n]

    # Normalize so that sum(|psi|^2) * dx = 1
    psi_int = psi_int / np.sqrt(np.sum(np.abs(psi_int) ** 2) * dx)

    # Reconstruct full wavefunction: pad with zeros at the two boundary points
    psi = np.zeros_like(x, dtype=float)
    psi[1:-1] = psi_int

    prob_density = np.abs(psi) ** 2

    plt.plot(x, prob_density, label=f"n={n + 1},  E={energies[n]:.5f}")

# Shade the classically allowed well region
plt.axvspan(
    left_wall, right_wall, alpha=0.12, color="steelblue", label="well region (V=0)"
)

# Mark the potential barrier level for reference
plt.axhline(0, color="black", linewidth=0.8)

plt.xlabel("Position $x$ (natural units)")
plt.ylabel(r"Probability density $|\psi_n(x)|^2$")
plt.title(
    f"Finite Square Well: First Four Bound States\n"
    f"($L={N_well}$, $V_0={V0}$, $\\hbar=m=\\Delta x=1$, "
    f"matrix size $N={N}$)"
)
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Reading the plot.**
Several features in the probability density plot directly reflect the
physics of the finite square well:

- **Quantum numbers:** state $n$ has exactly $n$ peaks inside the well,
  matching the node structure of the analytic solutions.
- **Barrier penetration:** the wavefunctions do not drop to zero at the
  well walls - they decay exponentially into the shaded region,
  reflecting the non-zero probability of finding the electron in the
  classically forbidden zone.
- **Energy ordering:** higher $n$ states have shorter spatial wavelengths
  inside the well (more oscillations in the same width), corresponding
  to larger kinetic energy and higher $E_n$.
- **Finite vs. infinite well:** the energies are slightly lower than the
  infinite-square-well values $E_n^\infty = n^2\pi^2\hbar^2/(2mL^2)$
  because the electron can spread into the barrier, effectively
  experiencing a larger box.

**What the matrix did:** the $129 \times 129$ Hamiltonian matrix encoded
the entire physical problem - the kinetic energy operator as a tridiagonal
coupling and the potential as a diagonal modification.
A single call to `np.linalg.eigh` solved all bound states simultaneously,
with no need to solve the transcendental equations that the analytic
approach requires.